# Analyzing Hubway Trips in SQL

In [1]:
#You will have to install ipython-sql using the following command:

!pip install jupysql

#Just Execute the cell and it will install

In [2]:
# If your file is in another Directory (Without Space in Name of Directory) : %sql sqlite:///Hello/Another_Folder/hubway_small_ok.db

# If your file is in Same Directory : %sql sqlite:///hubway_small_ok.db

%reload_ext sql
%sql sqlite:///hubway.db
%config SqlMagic.autocommit=False


Connecting to 'sqlite:///hubway.db'

**Please write your queries below in the empty boxes same as you write in a SQL Window**

Just append **%sql** before your query like this '%sql Select * from Trips limit 5;'

In [3]:
%%sql

Select * from Trips limit 5

Running query in 'sqlite:///hubway.db'

id,duration,start_date,start_station,end_date,end_station,bike_number,sub_type,zip_code,birth_date,gender
1,9,2011-07-28 10:12:00,23,2011-07-28 10:12:00,23,B00468,Registered,'97217,1976.0,Male
2,220,2011-07-28 10:21:00,23,2011-07-28 10:25:00,23,B00554,Registered,'02215,1966.0,Male
3,56,2011-07-28 10:33:00,23,2011-07-28 10:34:00,23,B00456,Registered,'02108,1943.0,Male
4,64,2011-07-28 10:35:00,23,2011-07-28 10:36:00,23,B00554,Registered,'02116,1981.0,Female
5,12,2011-07-28 10:37:00,23,2011-07-28 10:37:00,23,B00554,Registered,'97214,1983.0,Female


**About the Dataset and Schema**

The database has two tables, trips and stations. To begin with, we'll look at the trips table. It contains the following columns:


**Trips:**

id — A unique integer that serves as a reference for each trip

duration — The duration of the trip, measured in seconds

start_date — The date and time the trip began

start_station — An integer that corresponds to the id column in the stations table for the station the trip started at

end_date — The date and time the trip ended

end_station — The 'id' of the station the trip ended at

bike_number — Hubway's unique identifier for the bike used on the trip

sub_type — The subscription type of the user. "Registered" for users with a membership, "Casual" for users without a membership

zip_code — The zip code of the user (only available for registered members)

birth_date — The birth year of the user (only available for registered members)

gender — The gender of the user (only available for registered members)


**Station:**


id — A unique identifier for each station (corresponds to the start_station and end_station columns in the trips table)

station — The station name

municipality — The municipality that the station is in (Boston, Brookline, Cambridge or Somerville)

lat — The latitude of the station

lng — The longitude of the station

## Analytical Questions Test in SQL

Q1. What was the duration of the longest trip? What was the average trip duration? What was the smallest trip duration?

In [4]:


%%sql
SELECT MIN(duration), MAX(duration), AVG(duration)
FROM trips;

Running query in 'sqlite:///hubway.db'

MIN(duration),MAX(duration),AVG(duration)
0,9999,912.4096819046612


Q2. How many trips were taken by 'Registered' users?

Hint: Use Sub Type to Filter by Where

In [5]:
%%sql

SELECT COUNT(*) AS registered_trips
FROM trips
WHERE sub_type = 'Registered';

Running query in 'sqlite:///hubway.db'

registered_trips
1105192


Q3. How many trips were taken by male users in comparison to Female users? for Registered users only

Hint: Group By Gender and Calculate Count

In [6]:
%%sql

SELECT gender, COUNT(*) AS trip_count
FROM trips
WHERE sub_type = 'Registered'
GROUP BY gender;

Running query in 'sqlite:///hubway.db'

gender,trip_count
Female,271333
Female,1
Male,833858


Q4. Do registered or casual users take longer trips?

Hint: Group By Sub Type and Calculate Average

In [7]:
%%sql

SELECT sub_type, AVG(duration) AS average_duration
FROM trips
GROUP BY sub_type;

Running query in 'sqlite:///hubway.db'

sub_type,average_duration
Casual,1519.6438967403815
Registered,657.0260669639302


Q5. Which bike was used for the most trips?

Hint : Use Bike Number and Limit Results to 1

In [8]:
%%sql

SELECT bike_number, COUNT(*) AS trip_count
FROM trips
GROUP BY bike_number
ORDER BY trip_count DESC
LIMIT 1;

Running query in 'sqlite:///hubway.db'

bike_number,trip_count
B00490,2120


Q6. What is the average duration of trips by users over the age of 30?

Hint : DoB is already in years and use Airthmetic operator to calculate Age like (2026-DOB)

Link : #GETTING AGE IN SQLLITE : https://stackoverflow.com/questions/49929953/how-to-calculate-approximate-age-based-on-birth-year

In [9]:
%%sql

SELECT AVG(duration) AS average_duration
FROM trips
WHERE sub_type = 'Registered'
  AND (2026 - birth_date) > 30;

Running query in 'sqlite:///hubway.db'

average_duration
657.8409386999451


Q7. Which stations are most frequently used for round trips?

Hint: Round trip is considered as where Start and Stop Stations are same. Limit to top 5 Stations

In [10]:
%%sql

SELECT start_station, COUNT(*) AS round_trip_count
FROM trips
WHERE start_station = end_station
GROUP BY start_station
ORDER BY round_trip_count DESC
LIMIT 5;

Running query in 'sqlite:///hubway.db'

start_station,round_trip_count
58,3064
36,2548
42,2163
53,2144
52,1636


Q8. How many trips start and end in different municipalities?

Hint: We need to JOIN the trips table to the stations table twice. Once ON the start_station column and then ON the end_station

In [11]:
%%sql

SELECT COUNT(*) AS different_municipality_trips
FROM trips AS t
JOIN stations AS s1
    ON t.start_station = s1.id
JOIN stations AS s2
    ON t.end_station = s2.id
WHERE s1.municipality <> s2.municipality;

Running query in 'sqlite:///hubway.db'

different_municipality_trips
309748


Q9. How many trips incurred additional fees (lasted longer than 30 minutes)?

Hint: Use an Arithmetic function to calculate Duration in Minutes and Use in Where Clause

In [12]:
%%sql

SELECT COUNT(*) AS trips_over_30_minutes
FROM trips
WHERE duration > (30 * 60);

Running query in 'sqlite:///hubway.db'

trips_over_30_minutes
123155


Q10. Which bike was used for the longest total time? Provide the answer in Hours

Hint: Use an Group by Bike Number to sum Duration in Seconds and Limit Result to 1

In [13]:
%%sql

SELECT bike_number,
       SUM(duration) / 3600.0 AS total_hours
FROM trips
GROUP BY bike_number
ORDER BY total_hours DESC
LIMIT 1;

Running query in 'sqlite:///hubway.db'

bike_number,total_hours
B00490,571.8452777777778


Q11. Did registered or casual users take more round trips?

Hint: Recall Q7 and Modify your query to answer by Sub Type

In [14]:
%%sql

SELECT sub_type, COUNT(*) AS round_trip_count
FROM trips
WHERE start_station = end_station
GROUP BY sub_type
ORDER BY round_trip_count DESC;

Running query in 'sqlite:///hubway.db'

sub_type,round_trip_count
Casual,41427
Registered,31635


Q12. Which municipality had the most frequent Station End?

Hint: Join the Station table on End Station and then Group by Municipality to find answer

In [15]:
%%sql

SELECT s.municipality, COUNT(*) AS end_trip_count
FROM trips AS t
JOIN stations AS s
    ON t.end_station = s.id
GROUP BY s.municipality
ORDER BY end_trip_count DESC
LIMIT 1;

Running query in 'sqlite:///hubway.db'

municipality,end_trip_count
Boston,1212364


Q13. Which From and To Routes are Frequent with Regards to Municipalities?

Hint: Get Start and End Municipalities and Group by Both Attributes for Count of Rides

In [16]:
%%sql

SELECT s1.municipality AS from_municipality,
       s2.municipality AS to_municipality,
       COUNT(*) AS trip_count
FROM trips AS t
JOIN stations AS s1
    ON t.start_station = s1.id
JOIN stations AS s2
    ON t.end_station = s2.id
GROUP BY s1.municipality, s2.municipality
ORDER BY trip_count DESC;

Running query in 'sqlite:///hubway.db'

from_municipality,to_municipality,trip_count
Boston,Boston,1081805
Cambridge,Cambridge,162538
Boston,Cambridge,110968
Cambridge,Boston,110078
Cambridge,Somerville,20998
Somerville,Cambridge,20765
Boston,Brookline,14838
Brookline,Boston,14573
Somerville,Somerville,12746
Boston,Somerville,6071


Q14. Which Age Band in "Registered Users" has the most Rides? Calculate Age Bands based on interval of 10 and Cut Off by 70 or Above

Hint: Use Case When to Create Range of Value Bands as per Above Criteria then Group By. Use DoB Years to Calculate Age First (https://youtu.be/G46GIIRbO-g?t=76)

In [17]:
%%sql

SELECT
    CASE
        WHEN (2026 - birth_date) < 20 THEN 'Under 20'
        WHEN (2026 - birth_date) < 30 THEN '20-29'
        WHEN (2026 - birth_date) < 40 THEN '30-39'
        WHEN (2026 - birth_date) < 50 THEN '40-49'
        WHEN (2026 - birth_date) < 60 THEN '50-59'
        WHEN (2026 - birth_date) < 70 THEN '60-69'
        ELSE '70+'
    END AS age_band,
    COUNT(*) AS ride_count
FROM trips
WHERE sub_type = 'Registered'
GROUP BY age_band
ORDER BY ride_count DESC;

Running query in 'sqlite:///hubway.db'

age_band,ride_count
70+,777279
40-49,145495
50-59,71365
30-39,59785
60-69,51268


# Analyzing Hubway Trips in Python

In [1]:
# MANDATORY CELL TO EXECUTE FIRST SO WE HAVE DATA IN THE RIGHT TYPE

import sqlite3

import pandas as pd, numpy as np

cnx = sqlite3.connect('hubway.db')

cnx.cursor().executescript("""

UPDATE trips SET duration       = NULL WHERE duration IS '';
UPDATE trips SET start_date     = NULL WHERE start_date IS '';
UPDATE trips SET start_station  = NULL WHERE start_station IS '';
UPDATE trips SET end_date       = NULL WHERE end_date IS '';
UPDATE trips SET end_station    = NULL WHERE end_station IS '';
UPDATE trips SET sub_type       = NULL WHERE sub_type IS '';
UPDATE trips SET zip_code       = NULL WHERE zip_code IS '';
UPDATE trips SET birth_date     = NULL WHERE birth_date IS '';
UPDATE trips SET gender         = NULL WHERE gender IS '';

""")

trips=pd.read_sql_query("SELECT * FROM trips", cnx)
stations=pd.read_sql_query("SELECT * FROM stations", cnx)

print(len(trips))
print(len(stations))

trips=trips\
    .replace(r'^\s*$', np.nan, regex=True)\
    .astype({\
            'start_date':'datetime64[ns]'\
            ,'end_date':'datetime64[ns]'\
            ,'start_station':pd.Int64Dtype()\
            ,'end_station':pd.Int64Dtype()\
            ,'birth_date':pd.Int64Dtype()\
            })

stations=stations\
    .replace(r'^\s*$', np.nan, regex=True)\
    .astype({\
            'lat':'float64'\
            ,'lng':'float64'\
            })

print(len(trips))
print(len(stations))

cnx.close()

1570001
142
1570001
142


In [2]:
# ONLY RUN IF YOU HAVE ERROR IN ABOVE CELL (LIKE CANNOT FIND SQLLITE3)

#You will have to install SQLLite Connector using the following command:

!pip install pysqlite3

#Just Execute the cell and it will install

#THEN USE import pysqlite3 as sqlite3 in the import as replacement

In [6]:
# CHECKING IF THE DATA IS LOAD INTO RIGHT TABLES IN PYTHON DF

print(trips.head(5))

print('\n ------- \n')

print(stations.head(5))

   id  duration          start_date  start_station            end_date  \
0   1         9 2011-07-28 10:12:00             23 2011-07-28 10:12:00   
1   2       220 2011-07-28 10:21:00             23 2011-07-28 10:25:00   
2   3        56 2011-07-28 10:33:00             23 2011-07-28 10:34:00   
3   4        64 2011-07-28 10:35:00             23 2011-07-28 10:36:00   
4   5        12 2011-07-28 10:37:00             23 2011-07-28 10:37:00   

   end_station bike_number    sub_type zip_code  birth_date  gender  
0           23      B00468  Registered   '97217        1976    Male  
1           23      B00554  Registered   '02215        1966    Male  
2           23      B00456  Registered   '02108        1943    Male  
3           23      B00554  Registered   '02116        1981  Female  
4           23      B00554  Registered   '97214        1983  Female  

 ------- 

   id                             station municipality        lat        lng
0   3              Colleges of the Fenway     

In [7]:
# EXAMPLE SOLUTION, REMEMBER YOUR ANSWER SHOULD BE PRECISE AND ON POINT (SORTED AND IN RIGHT FORMAT)

##PYTHON SOLUTION

trips.agg({'duration': ['mean', 'min', 'max']}).T

,mean,min,max
duration,912.409682,0.0,9999.0


In [8]:
trips[trips['sub_type'] == 'Registered'].shape[0]

1105192

In [9]:
registered = trips[trips['sub_type'] == 'Registered']

registered.groupby('gender').size()

gender
Female      271333
Female\n         1
Male        833858
dtype: int64

In [10]:
trips.groupby('sub_type')['duration'].mean()

sub_type
Casual        1519.643897
Registered     657.026067
Name: duration, dtype: float64

In [ ]:
trips['bike_number'].value_counts().head(1)

In [11]:
registered = trips[trips['sub_type'] == 'Registered'].copy()

registered['age'] = 2026 - registered['birth_date']
registered[registered['age'] > 30]['duration'].mean()

np.float64(657.8409386999451)

In [12]:
round_trips = trips[trips['start_station'] == trips['end_station']]

round_trips['start_station'].value_counts().head(5)

start_station
58    3064
36    2548
42    2163
53    2144
52    1636
Name: count, dtype: Int64

In [13]:
start = stations[['id', 'municipality']].rename(
    columns={'id': 'start_station', 'municipality': 'start_municipality'}
)

end = stations[['id', 'municipality']].rename(
    columns={'id': 'end_station', 'municipality': 'end_municipality'}
)
merged = trips.merge(start, on='start_station')
merged = merged.merge(end, on='end_station')
merged[merged['start_municipality'] != merged['end_municipality']].shape[0]

309748

In [14]:
trips[trips['duration'] > 30 * 60].shape[0]

123155

In [15]:
bike_time = trips.groupby('bike_number')['duration'].sum()

bike_time_hours = bike_time / 3600

bike_time_hours.sort_values(ascending=False).head(1)

bike_number
B00490    571.845278
Name: duration, dtype: float64

In [16]:
round_trips.groupby('sub_type').size().sort_values(ascending=False)

sub_type
Casual        41427
Registered    31635
dtype: int64

In [17]:
end_trips = trips.merge(
    stations[['id', 'municipality']],
    left_on='end_station',
    right_on='id'
)
end_trips['municipality'].value_counts().head(1)

municipality
Boston    1212364
Name: count, dtype: int64

In [18]:
routes = (
    merged
    .groupby(['start_municipality', 'end_municipality'])
    .size()
    .sort_values(ascending=False)
)

routes.head(4)

start_municipality  end_municipality
Boston              Boston              1081805
Cambridge           Cambridge            162538
Boston              Cambridge            110968
Cambridge           Boston               110078
dtype: int64

In [19]:
registered = trips[trips['sub_type'] == 'Registered'].copy()

registered['age'] = 2026 - registered['birth_date']

registered['age_band'] = pd.cut(
    registered['age'],
    bins=[0, 20, 30, 40, 50, 60, 70, float('inf')],
    labels=['Under 20', '20-29', '30-39', '40-49', '50-59', '60-69', '70+'],
    right=False
)

registered['age_band'].value_counts()
registered['age_band'].value_counts().head(1)

age_band
40-49    145495
Name: count, dtype: int64

### BONUS PYTHON QUESTION (10 Marks)

##### Can you show a Report in Python "What are the Top 5 States with Most Number of Rides" For Registered Users Only

##### Hints :

1. Use the Sampled Dataframe for your working, else it would take a lot of time to convert many rows in Original table.

2. You will need Data Type Conversion and Cleaning of Zip Code (Don't fear errors / Learn from them)

3. Where do you get the States? READ THE CELL BELOW

!pip install zipcodes

print('\n')

import zipcodes

##### Gets All the Information Available against ZipCode

print(zipcodes.matching('77429')[0].keys())

['zip_code', 'zip_code_type', 'active', 'city', 'acceptable_cities', 'unacceptable_cities', 'state', 'county', 'timezone', 'area_codes', 'world_region', 'country', 'lat', 'long']

##### Sample Working how to Fetch a Against a ZipCode

zipcodes.matching('77429')[0].get('state')

READ MORE > https://github.com/seanpianka/Zipcodes


In [20]:
#EXECUTE THIS CELL TO CREATE SAMPLED DATAFRAME

registered_trips_sampled=trips.query("sub_type=='Registered'").groupby('zip_code').apply(lambda x: x.sample(10, replace=True)).reset_index()

In [21]:
!pip install zipcodes

In [22]:
## YOUR REPORT SOLUTION IN PYTHON
import zipcodes
# USE registered_trips_sampled Dataframe and temporarily work on column zip_code by storing in variable 'x'
x = registered_trips_sampled['zip_code']

# CLEAN zip_code COLUMN FROM PUNCTUATIONS
x = x.astype(str).str.replace(r'[^0-9]', '', regex=True)

# REPLACE NAN VALUES WITH STRING '0' FOR API TO WORK
x = x.replace('', '00000')
x = x.fillna('00000')
x = x.str.zfill(5)   # pads anything shorter than 5 digits with leading zeros

# TRY TO USE LAMBDA FUNCTION IN PYTHON TO GET STATE FOR EACH ROW, ONCE YOUR ARE DONE WITH ABOVE STEPS
def get_state(z):
    match = zipcodes.matching(z)
    return match[0].get('state') if match else np.nan

x = x.apply(lambda z: get_state(z))
# SAVE THE OUTPUT INTO A COLUMN NAMED AS registered_trips_sampled['state']

registered_trips_sampled['state'] = x

In [23]:
# WE WILL USE THIS DICTIONARY TO MAP STATE NAMES INSTEAD OF ABBREVIATIONS

state_names = {
        'AK': 'Alaska',
        'AL': 'Alabama',
        'AR': 'Arkansas',
        'AS': 'American Samoa',
        'AZ': 'Arizona',
        'CA': 'California',
        'CO': 'Colorado',
        'CT': 'Connecticut',
        'DC': 'District of Columbia',
        'DE': 'Delaware',
        'FL': 'Florida',
        'GA': 'Georgia',
        'GU': 'Guam',
        'HI': 'Hawaii',
        'IA': 'Iowa',
        'ID': 'Idaho',
        'IL': 'Illinois',
        'IN': 'Indiana',
        'KS': 'Kansas',
        'KY': 'Kentucky',
        'LA': 'Louisiana',
        'MA': 'Massachusetts',
        'MD': 'Maryland',
        'ME': 'Maine',
        'MI': 'Michigan',
        'MN': 'Minnesota',
        'MO': 'Missouri',
        'MP': 'Northern Mariana Islands',
        'MS': 'Mississippi',
        'MT': 'Montana',
        'NA': 'National',
        'NC': 'North Carolina',
        'ND': 'North Dakota',
        'NE': 'Nebraska',
        'NH': 'New Hampshire',
        'NJ': 'New Jersey',
        'NM': 'New Mexico',
        'NV': 'Nevada',
        'NY': 'New York',
        'OH': 'Ohio',
        'OK': 'Oklahoma',
        'OR': 'Oregon',
        'PA': 'Pennsylvania',
        'PR': 'Puerto Rico',
        'RI': 'Rhode Island',
        'SC': 'South Carolina',
        'SD': 'South Dakota',
        'TN': 'Tennessee',
        'TX': 'Texas',
        'UT': 'Utah',
        'VA': 'Virginia',
        'VI': 'Virgin Islands',
        'VT': 'Vermont',
        'WA': 'Washington',
        'WI': 'Wisconsin',
        'WV': 'West Virginia',
        'WY': 'Wyoming'
}


In [24]:
   registered_trips_sampled.columns

Index(['zip_code', 'level_1', 'id', 'duration', 'start_date', 'start_station',
       'end_date', 'end_station', 'bike_number', 'sub_type', 'birth_date',
       'gender', 'state'],
      dtype='str')

In [28]:
# CREATE ANOTHER COLUMN BASED ON STATE DERIVED TO GET STATE NAMES USING ABOVE DICTIONARY USING MAP FUNCTION, SAVE in STATE_NAME

registered_trips_sampled['state_name'] = registered_trips_sampled['state'].map(state_names)
registered_trips_sampled

,zip_code,level_1,id,duration,start_date,start_station,end_date,end_station,bike_number,sub_type,birth_date,gender,state,state_name
0,'00210,746405,753511,360,2013-05-05 17:01:00,22,2013-05-05 17:07:00,40,T01399,Registered,<NA>,Male,NaN,NaN
1,'00210,1309279,1317815,900,2013-09-20 15:31:00,112,2013-09-20 15:46:00,40,B00443,Registered,<NA>,Male,NaN,NaN
2,'00210,259472,264454,417,2012-06-08 16:17:00,35,2012-06-08 16:24:00,40,B00312,Registered,1978,Male,NaN,NaN
3,'00210,715607,722561,480,2013-04-26 08:09:00,6,2013-04-26 08:17:00,40,T01269,Registered,<NA>,Male,NaN,NaN
4,'00210,853863,861287,300,2013-06-08 16:37:00,22,2013-06-08 16:42:00,47,B00291,Registered,<NA>,Male,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5295,2215,480944,487129,1190,2012-09-09 13:57:00,19,2012-09-09 14:17:00,13,T01429,Registered,1980,Male,MA,Massachusetts
5296,2215,479864,486042,1249,2012-09-08 22:50:00,39,2012-09-08 23:11:00,19,B00206,Registered,1980,Male,MA,Massachusetts
5297,2215,495551,501792,1149,2012-09-13 23:37:00,39,2012-09-13 23:56:00,19,T01040,Registered,1980,Male,MA,Massachusetts
5298,2215,469298,475427,960,2012-09-05 00:54:00,39,2012-09-05 01:10:00,32,B00415,Registered,1980,Male,MA,Massachusetts


In [26]:
# USE THE NEWLY CREATED COLUMN TO MAKE A SUMMARY REPORT FOR EACH STATE NAME BASED ON NUMBER OF TRIPS AND GET TOP 5

report = (
    registered_trips_sampled
    .dropna(subset=['state_name'])
    .groupby('state_name')
    .size()
    .reset_index(name='number_of_trips')
    .sort_values('number_of_trips', ascending=False)
    .head(5)
)

report

,state_name,number_of_trips
11,Massachusetts,3080
14,New Hampshire,460
16,New York,200
21,Rhode Island,190
3,Connecticut,160


### WE HOPE YOU ENJOYED LEARNING SQL & PYTHON TOGHETHER